# When Did Congress Start Talking About China?

In this notebook you'll dig through **35 years of congressional floor
speeches** (1990–2024) and watch a story emerge from raw counts: when Congress
started talking about China, what words changed around it, and *who* drove the surge.

**No coding experience needed.** Almost everything is pre-written. In marked spots
you'll edit a line of real Python (a word, a number, or a list) and watch the
picture change.

---

### ⚠️ Step zero — this is YOUR copy

This notebook is your personal copy — it has your name on the file. Your edits save
automatically; nobody else is working in it. If it ever gets hopelessly tangled,
**Ctrl+Z / Cmd+Z inside a cell fixes almost everything** — and if it doesn't, email
Heagen for a fresh copy.

### How this notebook works

- Cells run **top to bottom**. Click a cell, then press **Shift+Enter** (or ▶) to run it.
- **▶️ JUST RUN** cells: run them, don't edit them.
- **✏️ YOUR TURN** cells: change the line marked `# <-- EDIT`.
- Under every ✏️ cell is a **"What you should see"** box for checking yourself.

**Time:** built for ~4 hours. Rough guide: Setup + meet the data ~30 min ·
Beat 1 ~60 min · Beat 2 ~60 min · Beat 3 ~45 min · Stretch ~45 min.

## Setup

**▶️ JUST RUN:** loads our tools and the three datasets. (Everything here comes
pre-installed in Colab, so this is quick.)

In [ ]:
# ▶️ JUST RUN — imports and data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = "https://raw.githubusercontent.com/YOUR-ORG/trade-war-lab-notebooks/main/data"
mentions   = pd.read_csv(f"{BASE}/congress_mentions.csv")
companions = pd.read_csv(f"{BASE}/congress_companions.csv")
senators   = pd.read_csv(f"{BASE}/congress_senators.csv")

print(f"Loaded {len(mentions)} year-term counts, {len(companions)} companion words, "
      f"{len(senators)} senator-year counts. ✅")
mentions.head()

> **What you should see:** a line saying `Loaded 280 year-term counts, 40 companion
> words, 106 senator-year counts. ✅` and a small table with columns `year`, `term`,
> `mentions`.

Each row of `mentions` answers one question: *in year X, how many times did a floor
speech mention term Y?* Which terms can we look up? **▶️ JUST RUN:**

In [ ]:
# ▶️ JUST RUN — what terms are available?
sorted(mentions["term"].unique())

> **What you should see:** a list of 8 terms:
> `['china', 'climate', 'japan', 'russia', 'taiwan', 'tariff', 'terrorism', 'trade']`

One more setup cell, the plotting helpers the beats use. **▶️ JUST RUN:**

In [ ]:
# ▶️ JUST RUN — helper tools
def plot_terms(terms, start_year, end_year):
    """Plot yearly mention counts for one or more terms."""
    if isinstance(terms, str):
        terms = [terms]  # forgives a bare string instead of a list
    start_year, end_year = int(start_year), int(end_year)
    fig, ax = plt.subplots(figsize=(10, 5))
    for term in terms:
        d = mentions[(mentions.term == term.lower()) &
                     (mentions.year >= start_year) & (mentions.year <= end_year)]
        if len(d) == 0:
            print(f"('{term}' isn't in this dataset — available: {sorted(mentions.term.unique())})")
            continue
        ax.plot(d.year, d.mentions, marker="o", markersize=3, label=term)
        print(f"'{term}': {d.mentions.iloc[0]} mentions in {d.year.iloc[0]} → "
              f"{d.mentions.iloc[-1]} in {d.year.iloc[-1]} "
              f"(peak: {d.mentions.max()} in {d.loc[d.mentions.idxmax(), 'year']})")
    ax.set_xlabel("year"); ax.set_ylabel("floor-speech mentions per year")
    ax.legend(); ax.grid(alpha=0.3)
    ax.set_title("How often Congress says it")
    plt.tight_layout(); plt.show()

def compare_eras(era_a, era_b):
    """Side-by-side: which words appear next to 'China' in two different decades."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=False)
    for ax, era in zip(axes, [era_a, era_b]):
        d = companions[companions.era == era].sort_values("count")
        if len(d) == 0:
            print(f"(No era called '{era}' — options: {sorted(companions.era.unique())})")
            plt.close(fig); return
        ax.barh(d.word, d["count"])
        ax.set_title(f'Words next to "China" in the {era}')
        print(f"{era} top three: {', '.join(d.sort_values('count', ascending=False).word.head(3))}")
    plt.tight_layout(); plt.show()

def plot_senator(name):
    """One senator's China mentions per year vs. the Senate average."""
    d = senators[senators.senator.str.lower() == name.lower()]
    if len(d) == 0:
        print(f"('{name}' isn't in this dataset — options: {sorted(senators.senator.unique())})")
        return
    avg = senators[senators.senator == "Senate average"]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(d.year, d.mentions, marker="o", label=d.senator.iloc[0])
    ax.plot(avg.year, avg.mentions, linestyle="--", color="gray", label="Senate average")
    ax.set_xlabel("year"); ax.set_ylabel('floor mentions of "China"')
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
    peak = d.loc[d.mentions.idxmax()]
    print(f"{d.senator.iloc[0]} peaked at {peak.mentions} mentions in {peak.year}.")

def leaderboard(year):
    """Who said 'China' most in a given year?"""
    year = int(year)
    d = senators[(senators.year == year) & (senators.senator != "Senate average")]
    if len(d) == 0:
        print(f"(No data for {year} — this table covers {senators.year.min()}–{senators.year.max()}.)")
        return
    return d.sort_values("mentions", ascending=False)[["senator", "mentions"]].reset_index(drop=True)

def plot_ratio(term_a, term_b):
    """Stretch helper: the ratio of two terms' mentions over time."""
    a = mentions[mentions.term == term_a.lower()].set_index("year").mentions
    b = mentions[mentions.term == term_b.lower()].set_index("year").mentions
    ratio = (a / b).dropna()
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(ratio.index, ratio.values, marker="o", markersize=3)
    ax.axhline(1, color="gray", linestyle="--")
    ax.set_ylabel(f'"{term_a}" mentions ÷ "{term_b}" mentions')
    ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
    crossed = ratio[ratio > 1]
    if len(crossed) > 0:
        print(f'"{term_a}" first overtook "{term_b}" in {crossed.index[0]}.')

print("Helpers loaded ✅ — on to Beat 1!")

---
## Beat 1 — The curve ✏️

One line answers the whole notebook's title question. Let's draw it.

**✏️ YOUR TURN:** run the cell as-is first. Then change the terms in the list and
the year range, and re-run. Each run is one experiment.

In [ ]:
terms_to_plot = ["china"]      # <-- EDIT this list (try adding "japan" or "russia")
start_year    = 1990           # <-- EDIT
end_year      = 2024           # <-- EDIT

plot_terms(terms_to_plot, start_year, end_year)

> **What you should see** (with the starting values): a line chart with one rising
> line, and a printed summary like
> `'china': 210 mentions in 1990 → 1048 in 2024 (peak: ...)`
> The line is mostly flat-ish for years, has a visible bump around **2000**, and
> then climbs steeply after **2017**.
> **If you get a red error:** every term needs its own quotes with commas between —
> `["china", "japan"]` — and years are plain numbers, no quotes.

### Things to try

1. **The bump around 2000.** Zoom in: set the years to 1996–2004. Something specific
   happened in Congress in 1999–2000 that made everyone talk about China. Guess before
   you look it up. (Answer: the vote to grant China *Permanent Normal Trade Relations*
   on its way into the WTO, one of the biggest trade debates in congressional history.)
2. **Add `"japan"` to the list** and look at the early 1990s. Japan was the China of
   the '90s — trade fights, fear of being overtaken, all of it. Then it faded. Worth
   asking: will the China line ever do what the Japan line did?
3. **Add `"russia"`.** Its spikes are sharp and event-shaped (2014, 2022). Compare the
   *shape* of the Russia line to the China line: one spikes on events, the other
   climbs and stays. What does that difference tell you about how Congress sees each?

**A question to carry into Beat 2:** the curve tells you Congress talks about China
*more*. It cannot tell you whether they're saying the same things louder, or saying
something new. For that, you need the words *around* the word.

---
## Beat 2 — The words around the word ✏️

For each decade, we counted which words most often appear in the **same sentence**
as "China" in floor speeches. This is where the story lives.

**✏️ YOUR TURN:** pick two eras to compare. Options: `"1990s"`, `"2000s"`, `"2010s"`, `"2020s"`.

In [ ]:
era_one = "1990s"   # <-- EDIT
era_two = "2020s"   # <-- EDIT

compare_eras(era_one, era_two)

> **What you should see** (with the starting eras): two horizontal bar charts side by
> side, plus printed lines like
> `1990s top three: trade, most-favored-nation, human rights` and
> `2020s top three: CCP, threat, Taiwan`.
> **If you get a red error:** era names need quotes and must match exactly —
> `"1990s"`, not `1990s` or `"90s"`.

### Things to try

1. Walk decade by decade: 1990s→2000s, 2000s→2010s, 2010s→2020s. In your own words,
   write a one-sentence caption for each transition. (Example for the first: "China
   went from a trade question to a jobs question.")
2. Track single words across the decades. "Taiwan" appears in the 1990s, nearly
   vanishes, then comes roaring back. What story does that one word tell?
3. Worth noticing: **no human read these speeches.** A program
   counted word co-occurrences, and a genuine historical narrative fell out: the
   vocabulary shift from *trade partner* to *threat*. This is what data science looks
   like when it's aimed at your subjects — history, politics, rhetoric.

**Handoff:** the vocabulary changed. But an average hides individuals. Did all of
Congress change together, or did a few people drag the average? Beat 3.

---
## Beat 3 — Who's driving? ✏️

Aggregates hide people. Let's look at individual senators.

First, the leaderboard. **▶️ JUST RUN:**

In [ ]:
# ▶️ JUST RUN — who said "China" most in 2024?
leaderboard(2024)

> **What you should see:** a ranked table of senators. A few names sit far above
> the rest.

Now pick one and look at their whole trajectory.

**✏️ YOUR TURN:** available names are in the leaderboard above.

In [ ]:
senator_name = "Marco Rubio"   # <-- EDIT (try "Tom Cotton", "Josh Hawley", "Chuck Schumer")

plot_senator(senator_name)

> **What you should see** (with the starting name): a line for the senator, a gray
> dashed line for the Senate average, and a printed peak like
> `Marco Rubio peaked at ... mentions in 20...`. The senator's line pulls away from
> the average sharply after ~2018.
> **If you get a red error:** the name needs quotes, spelled as in the leaderboard.

### Things to try

1. Compare **Tom Cotton** and **Chuck Schumer**. Both talk about China a lot, but
   one has been steadily elevated since the 2010s and one exploded after 2018. Which
   pattern suggests a long-held issue, and which suggests responding to a moment?
2. **Josh Hawley** only enters in 2019 (he joined the Senate that year), and enters
   *high*. What does it mean that new senators now arrive already talking about China?
3. Run `leaderboard(2012)` in a scratch cell (Insert → Code cell) and compare it to
   `leaderboard(2024)`. How much of the top five is the same people getting louder
   vs. new people showing up?

---
## Stretch goal (optional) — the crossover

When exactly did China overtake Japan in congressional attention? A ratio answers
that in one picture.

**✏️ YOUR TURN:**

In [ ]:
term_a = "china"   # <-- EDIT
term_b = "japan"   # <-- EDIT

plot_ratio(term_a, term_b)

> **What you should see:** a line crossing the gray dashed line (ratio = 1), and a
> printed year when `"china"` first overtook `"japan"`.

Now invent your own ratio. `"china"` vs `"russia"`: which rival owns congressional
attention, and does 2022 change it? `"tariff"` vs `"trade"`: when did the *tool*
start crowding out the *topic*? Write one sentence about what your ratio shows.

---
## What you just did

With a handful of edited lines, you:

- plotted 35 years of congressional attention and found the 2000 PNTR bump and the
  post-2017 surge,
- watched the vocabulary around "China" shift from *trade, WTO, most-favored-nation*
  to *CCP, threat, genocide*,
- and broke the aggregate apart to see which individual senators drive it.

**The bridge to Week 2:** that post-2017 surge isn't just speeches — it's
legislation. Congress now produces so many China-related bills that the lab built a
machine-learning model just to keep up with them. Next week, **you** label the bills
that train and test that model. You've now seen the wave; next week you help measure it.

**Before you close:** write down (1) your favorite one-sentence caption from Beat 2,
and (2) one thing the *data* couldn't tell you that a historian would want to know.
Bring both to the Week 2 kickoff.